# Monday: words, embeddings and LLM labels

The same r/AmItheAsshole comments, measured three ways: TF-IDF, embeddings and an LLM label.

Work with a partner. Write code in the cells that start with a comment; the check comments tell you what to expect. Make a copy named `week10_live_workspace_mine.ipynb`. The LLM labels are in a file, so you don't need a Gemini key today.

In [ ]:
import warnings
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore", category=FutureWarning)  # a harmless warning from one of the libraries
pd.set_option("display.max_colwidth", 300)  # show more of each text

## 1. Load the comments

Most comments start with a verdict: NTA (not the asshole) or YTA (you're the asshole). The cell keeps those comments, puts the verdict in its own column, and removes it from the text.

In [ ]:
comments = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/docs/interactives/data/week08/aita_replies.csv").rename(columns={"idstr": "id"})
comments["verdict"] = comments["body"].str.upper().str.extract(r"^(NTA|YTA)\b")[0]
comments = comments.dropna(subset=["verdict"]).reset_index(drop=True)
comments["text"] = (comments["body"]
                    .str.replace(r"\b(?:NTA|YTA|ESH|NAH|INFO)\b", "", regex=True, case=False)
                    .str.split().str.join(" ").str.lstrip(" .,:;!-–—"))
comments[["verdict", "text"]].head()  # Check: 1035 rows in comments

In [ ]:
# The share of comments with each verdict. Check: NTA 0.63, YTA 0.37

## 2. TF-IDF: which words go with each verdict?

TF-IDF gives each comment one weight per word. The average weights for YTA and for NTA comments show which words go with each verdict.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", min_df=15)  # words in at least 15 comments
weights = pd.DataFrame(tfidf.fit_transform(comments["text"]).toarray(), columns=tfidf.get_feature_names_out())
by_verdict = weights.groupby(comments["verdict"]).mean().T  # one row per word: its average weight in NTA and in YTA comments
by_verdict["ratio"] = by_verdict["YTA"] / by_verdict["NTA"]
by_verdict.shape  # Check: (220, 3)

In [ ]:
# Show the 10 words with the highest ratio (YTA) and the 10 with the lowest (NTA).
# Check: the highest are fun, look, knew, world, dude; the lowest are behavior, respect, able, change, won

In [ ]:
# The 6 words with the lowest ratio (NTA) and the 6 with the highest (YTA), as bars
by_ratio = by_verdict.sort_values("ratio")
selected = pd.concat([by_ratio.head(6), by_ratio.tail(6)])
selected[["NTA", "YTA"]].plot.barh(figsize=(6, 5), xlabel="average TF-IDF weight", ylabel="")  # Check: behavior has a long NTA bar and no YTA bar

What kind of comment do the YTA words suggest?

Your answer:

## 3. Embeddings: search by meaning

An embedding turns each comment into 384 numbers. Comments about similar things get similar numbers, so a search with a sentence also finds comments that use other words.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # the first run downloads the model, about 90 MB
embeddings = model.encode(comments["text"].tolist(), normalize_embeddings=True)
embeddings.shape  # Check: (1035, 384)

In [ ]:
def search(sentence, n=5):
    """The n comments closest to a sentence."""
    query = model.encode([sentence], normalize_embeddings=True)[0]
    found = comments.assign(similarity=embeddings @ query)
    return found.sort_values("similarity", ascending=False).head(n)[["similarity", "verdict", "text"]]

search("Your partner is controlling you.")  # Check: the second comment starts "your girlfriend is disregarding"

In [ ]:
# How many comments use the word "controlling"? Check: 6. Which of the five comments above would a word search miss?
comments["text"].str.contains("controlling", case=False).sum()

In [ ]:
# Search with a sentence about a topic from your project. Then count the comments that contain its main word.
# Check: the search finds comments without that word

Do the comments the search finds agree on whether the poster was wrong?

Your answer:

## 4. An LLM label: does the comment say the poster was wrong?

The verdict word was removed, so you can check any label against it. The file has 20 comments, 10 NTA and 10 YTA, and the labels of five methods:

- `word_search`: YTA if the comment uses words like *rude* or *should have*
- `classifier`: TF-IDF and logistic regression
- `gemini_short`: Gemini, asked "Label this reply NTA or YTA."
- `gemini_careful`: Gemini, with a longer instruction that defines NTA and YTA
- `gemini_tone`: Gemini, asked "Is this comment positive or negative?" Negative counts as YTA.

In [ ]:
reply_labels = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_reply_labels.csv")
METHODS = ["word_search", "classifier", "gemini_short", "gemini_careful", "gemini_tone"]
reply_labels[["verdict", "text"] + METHODS].head()  # Check: 20 rows in reply_labels

In [ ]:
# Print the accuracy of each method in METHODS.
# Check: word_search 0.55, classifier 0.35, gemini_short 0.8, gemini_careful 0.8, gemini_tone 0.6

### Two kinds of mistakes

Say you are looking for YTA comments.

- A **false positive** is an NTA comment labeled YTA.
- A **false negative** is a YTA comment labeled NTA.

Accuracy counts both kinds the same way.

*Hint: the false positives of method `m` are the rows where `(reply_labels[m] == "YTA") & (reply_labels["verdict"] == "NTA")`.*

In [ ]:
pd.crosstab(reply_labels["verdict"], reply_labels["gemini_careful"])  # rows: true verdict, columns: label

In [ ]:
# For each method, count the false positives and the false negatives.
# Check: gemini_short 0 and 4, gemini_careful 2 and 2, gemini_tone 7 and 1

In [ ]:
# The share of comments each method labels YTA. The true share is 0.5.
(reply_labels[METHODS] == "YTA").mean()

`gemini_short` and `gemini_careful` have the same accuracy. Which kind of mistake does each make?

Your answer:

## If you have time

### Precision and recall

For the label you are looking for, here YTA:

- **Precision**: of the comments labeled YTA, the share that are YTA.
- **Recall**: of the YTA comments, the share labeled YTA.

Compute both for each method in Part 4.

In [ ]:
# Precision and recall for YTA. Check: gemini_short 1.0 and 0.6, gemini_tone 0.56 and 0.9

### Sentiment with VADER

`gemini_tone` is sentiment. VADER gets sentiment from a word list: about 7,500 words, each with a score (*good* +1.9, *bad* −2.5). It adds up the scores in a text and gives a number from −1 (negative) to +1 (positive).

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download("vader_lexicon", quiet=True)  # the word list, downloaded once
sia = SentimentIntensityAnalyzer()
sia.polarity_scores("Good for you for sticking up for yourself!")  # Check: compound is 0.49

In [ ]:
# Score each comment, then label it like gemini_tone: negative counts as YTA
reply_labels["vader_score"] = reply_labels["text"].map(lambda text: sia.polarity_scores(text)["compound"])
reply_labels["vader"] = reply_labels["vader_score"].map(lambda score: "YTA" if score < 0 else "NTA")
reply_labels[["verdict", "vader_score", "vader", "gemini_tone"]].head()  # Check: the first score is 0.51

In [ ]:
# In how many of the 20 comments do vader and gemini_tone give the same label?
# Check: 10

In [ ]:
# The comments where the two sentiment methods disagree
reply_labels.loc[reply_labels["vader"] != reply_labels["gemini_tone"], ["verdict", "vader_score", "gemini_tone", "text"]]

Pick one comment where VADER and Gemini disagree. Which one gets the tone right?

Your answer:

### Can tone stand in for the verdict?

Count negative as YTA and check both sentiment methods against the verdict.

In [ ]:
# The accuracy of vader and of gemini_tone against the verdict.
# Check: 0.5 and 0.6

In [ ]:
# The NTA comments the sentiment prompt labeled YTA, with Gemini's reason
reply_labels.loc[(reply_labels["verdict"] == "NTA") & (reply_labels["gemini_tone"] == "YTA"), ["text", "gemini_tone_reason"]]

Why do these NTA comments sound negative?

Your answer: